In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 3-Class Disaster Triage: Fuzzy Support Vector Machine with Hyperplane (FSVM-Hyperplane) (`models/train_fuzzy_models.ipynb`)

Loads the emergency triage cohort from **`datasets/5v_cleandf.RData`** (~558,000 encounters with valid ESI) and executes the **Fuzzy Support Vector Machine with Hyperplane (FSVM-Hyperplane)** algorithm ported directly from [`Fuzzy-SVM/FUZZY SVM.ipynb`](file:///home/apt2736/PKM_RF/Fuzzy-SVM/FUZZY%20SVM.ipynb).

```mermaid
flowchart TD
    Raw["Raw 8 Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> BaseSVM["1. Base SVM Hyperplane Fitting (Extracts Decision Hyperplanes f(x))"]
    BaseSVM --> Dist["2. Calculate Hyperplane Distance d_hyp(x_i) = y_i * f(x_i)"]
    Dist --> FuzzyMem["3. Compute Fuzzy Membership s_i = func(d_hyp) * (N_min / N_c)"]
    FuzzyMem --> FSVM["4. Train Fuzzy SVM with Sample Weights w_i = C * s_i"]
    FSVM --> Eval["5. Full Holdout Test Set Evaluation: Recall, Specificity, Balanced Acc, ROC-AUC"]
    FSVM --> Viz["6. 3x3 Confusion Matrix & Fuzzy Membership Distribution Plots"]
```

### 🎯 3-Tier Disaster Triage Acuity Mapping
1. **`Tier 0: RED (ESI 1)`** ($y=0$): Immediate Resuscitation / Life Threat ($5,271$ visits, $\sim 0.94\%$).
2. **`Tier 1: YELLOW (ESI 2–3)`** ($y=1$): Emergent & Urgent conditions ($440,059$ visits, $\sim 78.86\%$).
3. **`Tier 2: GREEN (ESI 4–5)`** ($y=2$): Semi-urgent & Non-urgent conditions ($112,699$ visits, $\sim 20.19\%$).

### 📐 Mathematical Formulation of FSVM-Hyperplane
As defined in [`Fuzzy-SVM/FUZZY SVM.ipynb`](file:///home/apt2736/PKM_RF/Fuzzy-SVM/FUZZY%20SVM.ipynb) (`HYP_SVM` class):
1. **Hyperplane Distance**: The signed functional distance of sample $x_i$ to the class boundary is:
   $$d_{\text{hyp}}(x_i) = y_i \cdot f(x_i) = y_i \left( \sum_{j \in \text{SV}} \alpha_j y_j K(x_j, x_i) + b \right)$$
2. **Fuzzy Membership Function**: For sigmoidal / decaying membership (Type 2):
   $$\text{func}(x_i) = \frac{2}{1 + \exp(-\beta \cdot d_{\text{hyp}}(x_i))}$$
3. **Class Imbalance Scaling**: With imbalance ratio factor $r_c = \frac{N_{\text{min}}}{N_c}$:
   $$s_i = \text{func}(x_i) \cdot r_{y_i}$$
4. **Fuzzy Optimization**: Samples in deep overlap receive lower weights ($s_i \to 0$), preventing majority boundary noise from shifting the decision boundary into the resuscitation tier.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data, Partition (70/15/15 Stratified) & Impute + Scale
# ---------------------------------------------------------------------------
import os, json, pickle, warnings, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# Retrieve matrices from R environment with fallback to pyreadr
try:
    from rpy2.robjects import r
    raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
    esi_all     = np.array(r('esi_export'), dtype=np.int32)
except Exception:
    import pyreadr
    candidate_paths = [
        '../datasets/5v_cleandf.RData',
        'datasets/5v_cleandf.RData',
        '/kaggle/working/PKM_RF/datasets/5v_cleandf.RData',
        '/kaggle/input/5v-cleandf/5v_cleandf.RData'
    ]
    rdata_path = next(p for p in candidate_paths if os.path.exists(p))
    res = pyreadr.read_r(rdata_path)
    df_raw = res[list(res.keys())[0]]
    df_raw = df_raw[df_raw['esi'].notna()]
    gender = df_raw['gender'].apply(lambda x: 1 if str(x) == 'Male' else (0 if str(x) == 'Female' else np.nan)).values
    cc_bd = df_raw['cc_breathingdifficulty'].values if 'cc_breathingdifficulty' in df_raw.columns else np.full(len(df_raw), np.nan)
    raw_mat_all = np.column_stack([
        df_raw['age'].values,
        cc_bd,
        gender,
        df_raw['triage_vital_hr'].values,
        df_raw['triage_vital_sbp'].values,
        df_raw['triage_vital_dbp'].values,
        df_raw['triage_vital_rr'].values,
        df_raw['triage_vital_o2'].values
    ]).astype(np.float64)
    esi_all = df_raw['esi'].astype(int).values

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Construct 3-Tier Target Variable:
# 0 = RED (ESI 1), 1 = YELLOW (ESI 2-3), 2 = GREEN (ESI 4-5)
y_all = np.zeros(len(esi_all), dtype=np.int32)
y_all[esi_all == 1] = 0
y_all[np.isin(esi_all, [2, 3])] = 1
y_all[np.isin(esi_all, [4, 5])] = 2
TIER_LABELS = ['RED (ESI 1)', 'YELLOW (ESI 2-3)', 'GREEN (ESI 4-5)']

print("=" * 75)
print("  3-TIER DISASTER TRIAGE COHORT (5v_cleandf.RData)")
print("=" * 75)
print(f"Total Valid ESI Encounters: {len(esi_all):,}")
for c, lbl in enumerate(TIER_LABELS):
    count_c = np.sum(y_all == c)
    print(f"  * Tier {c} [{lbl:<17}]: {count_c:>7,} ({count_c/len(y_all)*100:.2f}%)")
print("=" * 75 + "\n")

# Stratified 70% Train / 15% Validation / 15% Test Split
itr, itmp = train_test_split(np.arange(len(esi_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite  = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

y_train, y_val, y_test = y_all[itr], y_all[iva], y_all[ite]

# Median Imputation + Standard Scaling fitted strictly on Training partition
imputer   = SimpleImputer(strategy='median')
X_tr_imp  = imputer.fit_transform(raw_mat_all[itr])
X_val_imp = imputer.transform(raw_mat_all[iva])
X_te_imp  = imputer.transform(raw_mat_all[ite])

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_imp)
X_val   = scaler.transform(X_val_imp)
X_test  = scaler.transform(X_te_imp)

print(f"Partition Dimensions:")
print(f"  * Training Set  : {X_train.shape[0]:,} visits (70%)")
print(f"  * Validation Set: {X_val.shape[0]:,} visits (15%)")
print(f"  * Holdout Test  : {X_test.shape[0]:,} visits (15%)")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Fuzzy SVM with Hyperplane Algorithm Implementation
#
# Mathematical Reference: Fuzzy-SVM/FUZZY SVM.ipynb (HYP_SVM class)
#   1. Base SVM fit -> decision function values d_hyp(x_i) = y_i * f(x_i)
#   2. Fuzzy membership calculation:
#      - Linear (Type 1): func = 1 - (d_hyp / (max(d_hyp) + eps))
#      - Sigmoid/Exponential (Type 2): func = 2 / (1 + exp(-beta * d_hyp))
#   3. Class imbalance scaling: s_i = func(x_i) * (N_min / N_c)
#   4. Final Weighted SVM fit with sample_weight = s_i
# ---------------------------------------------------------------------------

class FSVM_Hyperplane:
    """
    Fuzzy Support Vector Machine with Hyperplane-Based Membership (FSVM-Hyperplane).
    Ported from Fuzzy-SVM/FUZZY SVM.ipynb.
    """
    def __init__(self, C=10.0, kernel='rbf', gamma='scale', beta=0.2,
                 membership_type='sigmoid', base_subsample=30000, random_state=42):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.beta = beta
        self.membership_type = membership_type
        self.base_subsample = base_subsample
        self.random_state = random_state
        
        self.base_svm_ = None
        self.fuzzy_svm_ = None
        self.sample_weights_ = None
        self.classes_ = None

    def _compute_hyperplane_memberships(self, X, y):
        """
        Step 1 & 2: Trains base SVM to determine hyperplane distances,
        then computes fuzzy memberships scaled by class imbalance ratio.
        """
        N = len(y)
        self.classes_, class_counts = np.unique(y, return_counts=True)
        n_min = np.min(class_counts)
        r_weights = {c: n_min / class_counts[idx] for idx, c in enumerate(self.classes_)}

        # Subsample for fast base hyperplane estimation if dataset is massive
        if self.base_subsample is not None and N > self.base_subsample:
            print(f"  [FSVM] Subsampling {self.base_subsample:,} points for Base Hyperplane Estimation...")
            sub_idx, _ = train_test_split(
                np.arange(N), train_size=self.base_subsample,
                stratify=y, random_state=self.random_state
            )
            X_base_fit, y_base_fit = X[sub_idx], y[sub_idx]
        else:
            X_base_fit, y_base_fit = X, y

        print("  [FSVM] Step 1: Fitting Base SVM on hyperplane geometry...")
        self.base_svm_ = SVC(
            C=self.C, kernel=self.kernel, gamma=self.gamma,
            decision_function_shape='ovr', random_state=self.random_state
        )
        self.base_svm_.fit(X_base_fit, y_base_fit)

        print("  [FSVM] Step 2: Computing signed hyperplane distances d_hyp...")
        # Decision values: distance to hyperplanes
        decision_values = self.base_svm_.decision_function(X)

        if len(self.classes_) == 2:
            y_signed = np.where(y == self.classes_[1], 1.0, -1.0)
            d_hyp = y_signed * decision_values
        else:
            # Multiclass OvR: signed distance of sample to its true class hyperplane
            d_hyp = np.zeros(N, dtype=np.float64)
            for idx, c in enumerate(self.classes_):
                c_mask = (y == c)
                d_hyp[c_mask] = decision_values[c_mask, idx]

        print(f"  [FSVM] Step 3: Computing fuzzy memberships (type='{self.membership_type}')...")
        if self.membership_type == 'linear':
            d_max = np.max(d_hyp) + 1e-6
            func = 1.0 - (d_hyp / d_max)
        elif self.membership_type == 'sigmoid':
            # Sigmoidal membership: 2 / (1 + exp(-beta * d_hyp))
            func = 2.0 / (1.0 + np.exp(-self.beta * d_hyp))
        else:
            # Rational decay: 2 / (1 + beta * max(0, d_hyp))
            func = 2.0 / (1.0 + self.beta * np.maximum(0, d_hyp))

        # Scale by class imbalance ratio r_c = N_min / N_c
        s = np.zeros(N, dtype=np.float64)
        for idx, c in enumerate(self.classes_):
            c_mask = (y == c)
            s[c_mask] = func[c_mask] * r_weights[c]

        # Ensure strictly positive weights and normalize mean to 1.0
        s = np.clip(s, 0.01, 10.0)
        s = s / np.mean(s)
        return s

    def fit(self, X, y, subsample_final=None):
        """
        Fits the Fuzzy Support Vector Machine using computed sample weights.
        """
        self.sample_weights_ = self._compute_hyperplane_memberships(X, y)
        
        fit_idx = np.arange(len(y))
        if subsample_final is not None and len(y) > subsample_final:
            print(f"  [FSVM] Subsampling {subsample_final:,} points for Final Weighted SVM fit...")
            fit_idx, _ = train_test_split(
                np.arange(len(y)), train_size=subsample_final,
                stratify=y, random_state=self.random_state
            )

        print("  [FSVM] Step 4: Fitting Final Fuzzy SVM with sample weights...")
        self.fuzzy_svm_ = SVC(
            C=self.C, kernel=self.kernel, gamma=self.gamma,
            probability=True, random_state=self.random_state
        )
        self.fuzzy_svm_.fit(X[fit_idx], y[fit_idx], sample_weight=self.sample_weights_[fit_idx])
        print("✓ FSVM-Hyperplane Training Complete!")
        return self

    def predict(self, X):
        return self.fuzzy_svm_.predict(X)

    def predict_proba(self, X):
        return self.fuzzy_svm_.predict_proba(X)

    def decision_function(self, X):
        return self.fuzzy_svm_.decision_function(X)


print("✓ FSVM_Hyperplane model class initialized successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Train FSVM-Hyperplane on the Disaster Triage Training Set
# ---------------------------------------------------------------------------
print("=" * 75)
print("  TRAINING FUZZY SVM WITH HYPERPLANE (FSVM-HYPERPLANE)")
print("=" * 75)

fsvm_model = FSVM_Hyperplane(
    C=10.0,
    kernel='rbf',
    gamma='scale',
    beta=0.2,
    membership_type='sigmoid',
    base_subsample=25000,
    random_state=42
)

t0 = time.time()
# Fit on training cohort (using 35k representative samples for quadratic solver scalability)
fsvm_model.fit(X_train, y_train, subsample_final=35000)
print(f"✓ Total FSVM-Hyperplane Training Time: {time.time()-t0:.1f}s\n")

# Quick Validation Check
pred_val = fsvm_model.predict(X_val)
val_bacc = balanced_accuracy_score(y_val, pred_val)
val_rec0 = recall_score(y_val, pred_val, labels=[0], average=None, zero_division=0)[0]
print(f"Validation Set Quick Check:")
print(f"  * Macro Balanced Accuracy : {val_bacc*100:.2f}%")
print(f"  * RED (ESI 1) Recall      : {val_rec0*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Comprehensive Holdout Test Evaluation
#
# Reports:
#   - Recall (Sensitivity) per tier
#   - Specificity (True Negative Rate) per tier
#   - Macro Balanced Accuracy
#   - Macro & Per-Class ROC-AUC (One-vs-Rest)
#   - 3x3 Confusion Matrix
# ---------------------------------------------------------------------------
print("Evaluating FSVM-Hyperplane on Holdout Test Set (83,705 visits)...")

t_eval = time.time()
pred_test = fsvm_model.predict(X_test)
p_test    = fsvm_model.predict_proba(X_test)
print(f"✓ Inference Completed in {time.time()-t_eval:.1f}s\n")

# 1. Overall & Macro Metrics
acc         = accuracy_score(y_test, pred_test)
bal_acc     = balanced_accuracy_score(y_test, pred_test)
macro_f1    = f1_score(y_test, pred_test, average='macro', zero_division=0)
weighted_f1 = f1_score(y_test, pred_test, average='weighted', zero_division=0)

# 2. Per-Class Sensitivity (Recall), Precision, F1
recall_per = recall_score(y_test, pred_test, average=None, zero_division=0)
prec_per   = precision_score(y_test, pred_test, average=None, zero_division=0)
f1_per     = f1_score(y_test, pred_test, average=None, zero_division=0)

# 3. Per-Class Specificity (True Negative Rate)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2])
specificity_per = []
for c in range(3):
    tp = cm[c, c]
    fn = cm[c, :].sum() - tp
    fp = cm[:, c].sum() - tp
    tn = cm.sum() - tp - fn - fp
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificity_per.append(spec)
macro_spec = np.mean(specificity_per)

# 4. Per-Class & Macro ROC-AUC (One-vs-Rest)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
roc_auc_ovr = roc_auc_score(y_test_bin, p_test, average='macro', multi_class='ovr')
roc_auc_per = []
for c in range(3):
    auc_c = roc_auc_score(y_test_bin[:, c], p_test[:, c])
    roc_auc_per.append(auc_c)

# Detailed Summary Table
report_rows = []
for c, lbl in enumerate(TIER_LABELS):
    report_rows.append({
        'Triage_Tier': lbl,
        'True_Visits': int(np.sum(y_test == c)),
        'Predicted_Visits': int(np.sum(pred_test == c)),
        'Recall (Sensitivity)': f"{recall_per[c]*100:.2f}%",
        'Specificity': f"{specificity_per[c]*100:.2f}%",
        'Precision': f"{prec_per[c]*100:.2f}%",
        'F1_Score': round(f1_per[c], 4),
        'ROC_AUC (OvR)': round(roc_auc_per[c], 4)
    })

report_df = pd.DataFrame(report_rows)

print("=" * 105)
print("     HOLDOUT TEST EVALUATION: FSVM WITH HYPERPLANE (3-TIER DISASTER TRIAGE)")
print("=" * 105)
print(f"Total Test Encounters       : {len(y_test):,} visits")
print(f"Overall Accuracy            : {acc*100:.2f}%")
print(f"Macro Balanced Accuracy     : {bal_acc*100:.2f}%")
print(f"Macro Specificity           : {macro_spec*100:.2f}%")
print(f"Macro ROC-AUC (OvR)         : {roc_auc_ovr:.4f}")
print(f"Macro F1-Score              : {macro_f1:.4f}")
print(f"Weighted F1-Score           : {weighted_f1:.4f}")
print("-" * 105)
print(report_df.to_string(index=False))
print("=" * 105 + "\n")

print("Classification Report:")
print(classification_report(y_test, pred_test, target_names=TIER_LABELS, digits=4))

# Export Report CSV
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'fsvm_hyperplane_3class_report.csv')
report_df.to_csv(report_file, index=False)
print(f"✓ Report successfully saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 3x3 Confusion Matrix Visualization
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8.5, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Purples', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=TIER_LABELS, yticklabels=TIER_LABELS
)

ax.set_title(
    f'FSVM-Hyperplane: 3-Class Confusion Matrix\n'
    f'Balanced Accuracy: {bal_acc*100:.2f}% | Macro ROC-AUC: {roc_auc_ovr:.4f}',
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel('Predicted Disaster Triage Tier', fontsize=11, fontweight='bold')
ax.set_ylabel('True Disaster Triage Tier', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path1 = os.path.join(plots_dir, 'distant_analysis', 'fsvm_hyperplane_3class_confusion_matrix.png')
cm_path2 = os.path.join(plots_dir, 'image', 'fsvm_hyperplane_3class_confusion_matrix.png')
plt.savefig(cm_path1, dpi=300, bbox_inches='tight')
plt.savefig(cm_path2, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix saved to: {cm_path1}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Fuzzy Membership Weight Distribution Analysis
# ---------------------------------------------------------------------------
weights = fsvm_model.sample_weights_

fig, ax = plt.subplots(figsize=(10, 5))
palette = {'RED (ESI 1)': '#e74c3c', 'YELLOW (ESI 2-3)': '#f1c40f', 'GREEN (ESI 4-5)': '#2ecc71'}

plot_data = []
for c, lbl in enumerate(TIER_LABELS):
    c_weights = weights[y_train == c]
    for w in np.random.choice(c_weights, min(5000, len(c_weights)), replace=False):
        plot_data.append({'Tier': lbl, 'Fuzzy_Weight': w})

df_plot = pd.DataFrame(plot_data)
sns.boxplot(data=df_plot, x='Tier', y='Fuzzy_Weight', palette=palette, ax=ax, width=0.45)
ax.set_title('FSVM-Hyperplane: Learned Fuzzy Sample Weight Distribution per Tier', fontsize=12.5, fontweight='bold', pad=12)
ax.set_xlabel('Disaster Triage Tier', fontsize=11, fontweight='bold')
ax.set_ylabel('Fuzzy Sample Weight (s_i)', fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
mem_path = os.path.join(plots_dir, 'distant_analysis', 'fsvm_hyperplane_membership_distribution.png')
plt.savefig(mem_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Fuzzy membership distribution plot saved to: {mem_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'fsvm_model': fsvm_model,
    'features': FEATURES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'fsvm_hyperplane_3class_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='Fuzzy_SVM_with_Hyperplane_3Class',
    method='FSVM-Hyperplane (Sigmoid Distance Membership + Class Imbalance Scaling)',
    reference='Fuzzy-SVM/FUZZY SVM.ipynb (HYP_SVM)',
    classifier='FSVM_Hyperplane',
    C=float(fsvm_model.C),
    beta=float(fsvm_model.beta),
    kernel=str(fsvm_model.kernel),
    features=FEATURES,
    tier_labels=TIER_LABELS,
    total_samples=len(esi_all),
    holdout_test_samples=len(y_test),
    overall_accuracy=round(acc, 4),
    macro_balanced_accuracy=round(bal_acc, 4),
    macro_specificity=round(macro_spec, 4),
    macro_roc_auc_ovr=round(roc_auc_ovr, 4),
    macro_f1=round(macro_f1, 4),
    per_class_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'fsvm_hyperplane_3class_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Deployment Bundle  : {bundle_file}")
print(f"✓ Deployment Manifest: {manifest_file}")